# Phase 1 Week 1 — Synthea → MIMIC-IV Schema Mapping
### -- Work in Progress --

## Daun's Notes:

#### TODO Verify: The task appears to be to map the following: Synthea FHIR JSON →  MIMIC-IV-on-FHIR profiles
I interpret “MIMIC-IV schema-derived” as meaning that the synthetic event stream should be informed by the MIMIC-IV data model, especially through the published MIMIC-IV-on-FHIR profiles and mappings, rather than requiring us to recreate the full MIMIC-IV relational database.

**Make Synthea-generated FHIR look more like MIMIC-IV-on-FHIR.**

- I could not find a published Synthea FHIR-to-MIMIC-IV-on-FHIR mapping pipeline.
- Existing work mostly supports MIMIC-IV relational-to-FHIR conversion or general Synthea FHIR generation.

Use existing MIMIC-IV → FHIR mappings as the target specification,
then build our own Synthea FHIR → MIMIC-shaped data transformation.

- Synthea FHIR and MIMIC-IV-on-FHIR are not the same flavor of FHIR.
- Synthea creates synthetic patient records using common FHIR resources and standard clinical vocabularies.
- MIMIC-IV-on-FHIR uses MIMIC-specific profiles, local MIMIC terminology, and modeling choices designed to preserve the original MIMIC relational data.
- The MIMIC-IV-on-FHIR paper notes that the project intentionally modeled MIMIC-specific profiles and terminology, and did not fully map local MIMIC terminology to US Core or standard ontologies.

### Resources
- Synthea Notes: https://mitre.github.io/fhir-for-research/modules/synthea-overview
- MIMIC-IV on FHIR Notes: https://physionet.org/content/mimic-iv-fhir/2.1/


## Overview

Related to notebook `P1_Wk1_validation.ipynb`.

Answers *"what's in it, and how does it map to MIMIC-IV?"*

Two distinct mapping problems:

1. **Structural** — FHIR resource/field → MIMIC-IV table/column.
2. **Vocabulary** — SNOMED-CT / LOINC / RxNorm → ICD-9/10 / `itemid` / GSN/NDC. Concept normalization is its own research problem; placeholder § 3.

The cohort walk is duplicated from the validation notebook; in future, may want to factor `iter_bundles` into `src/pipeline/`.

Output: `references/synthea_to_mimiciv.md`.

### Prerequisite

Per README, Run `./scripts/generate_cohort.sh` once to populate `data/raw/synthea/fhir/`. The script uses Docker to run Synthea, but the output is written directly to the host filesystem via a bind mount — Docker is not needed at notebook runtime.

### Setup

In [1]:
import json
from collections import Counter, defaultdict
from pathlib import Path

FHIR_DIR = Path("../data/raw/synthea/fhir")

all_files = sorted(FHIR_DIR.glob("*.json"))
patient_files = [p for p in all_files if not p.name.startswith(("hospital", "practitioner"))]

print(f"Patient bundles: {len(patient_files)}")


def iter_bundles(paths=None):
    paths = patient_files if paths is None else paths
    for p in paths:
        with p.open(encoding="utf-8") as fh:
            yield p, json.load(fh)

Patient bundles: 1178


## 1. Field inventory

Synthea only emits a subset of FHIR R4, and which fields are populated varies. Rather than working from the FHIR spec, this section walks the cohort and records *every JSON path that actually appears* in each resource type, with frequency.

The output answers two questions for the schema-mapping work:

- **What fields exist?** (the rows themselves)
- **How reliably are they populated?** (the `pct` column — 100% means every resource of that type has it; 5% means it's rare and likely optional/edge-case)

Use `print_inventory(rtype, head=N, min_pct=P)` to inspect any resource type. Examples shown for `Patient` (small, universal fields) and `Encounter` (filtered to ≥10% to suppress noise).

The data also lives in `paths_per_type` (a `dict[str, Counter[str, int]]`) and `counts_per_type` for programmatic use.

In [2]:
def collect_paths(obj, prefix=""):
    """Yield every JSON path that appears in `obj`, collapsing list indices to `[]`."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            new = f"{prefix}.{k}" if prefix else k
            yield new
            yield from collect_paths(v, new)
    elif isinstance(obj, list):
        for item in obj:
            yield from collect_paths(item, f"{prefix}[]")


# Walk the full cohort, recording which paths appear in each resource of each type.
paths_per_type = defaultdict(Counter)
counts_per_type = Counter()

for _, bundle in iter_bundles():
    for entry in bundle["entry"]:
        r = entry["resource"]
        rtype = r["resourceType"]
        counts_per_type[rtype] += 1
        for path in set(collect_paths(r)):
            paths_per_type[rtype][path] += 1


def print_inventory(rtype, head=None, min_pct=0.0):
    """Print the field inventory for one resource type, sorted by frequency."""
    n = counts_per_type[rtype]
    if n == 0:
        print(f"No {rtype} resources found.")
        return
    print(f"{rtype}  ({n:,} resources)")
    print(f"{'pct':>6}  {'count':>8}  path")
    print("-" * 70)
    rows = sorted(paths_per_type[rtype].items(), key=lambda kv: (-kv[1], kv[0]))
    for path, count in rows:
        pct = 100 * count / n
        if pct < min_pct:
            continue
        if head and rows.index((path, count)) >= head:
            break
        print(f"  {pct:5.1f}%  {count:>8,}  {path}")


# Resource types sorted by count — the universe to map to MIMIC-IV.
print("Resource types in cohort (mapping targets):\n")
for rtype, n in counts_per_type.most_common():
    print(f"  {rtype:30s} {n:>8,}")

print("\n" + "=" * 70)
print_inventory("Patient")
print()
print_inventory("Encounter", min_pct=10.0)  # filter rare paths

Resource types in cohort (mapping targets):

  Observation                     662,643
  Procedure                       200,872
  DiagnosticReport                146,525
  Claim                           134,470
  ExplanationOfBenefit            134,470
  Encounter                        71,330
  DocumentReference                71,330
  MedicationRequest                63,140
  Condition                        44,368
  SupplyDelivery                   30,355
  Medication                       21,743
  MedicationAdministration         21,743
  Immunization                     16,998
  Device                            6,837
  ImagingStudy                      6,093
  CareTeam                          3,931
  CarePlan                          3,931
  Patient                           1,178
  Provenance                        1,178
  AllergyIntolerance                1,141

Patient  (1,178 resources)
   pct     count  path
----------------------------------------------------------------

In [3]:
print_inventory("Observation", head=40)   # top 40 paths


Observation  (662,643 resources)
   pct     count  path
----------------------------------------------------------------------
  100.0%   662,643  category
  100.0%   662,643  category[].coding
  100.0%   662,643  category[].coding[].code
  100.0%   662,643  category[].coding[].display
  100.0%   662,643  category[].coding[].system
  100.0%   662,643  code
  100.0%   662,643  code.coding
  100.0%   662,643  code.coding[].code
  100.0%   662,643  code.coding[].display
  100.0%   662,643  code.coding[].system
  100.0%   662,643  code.text
  100.0%   662,643  effectiveDateTime
  100.0%   662,643  encounter
  100.0%   662,643  encounter.reference
  100.0%   662,643  id
  100.0%   662,643  issued
  100.0%   662,643  resourceType
  100.0%   662,643  status
  100.0%   662,643  subject
  100.0%   662,643  subject.reference
   93.1%   617,039  meta
   93.1%   617,039  meta.profile
   77.7%   514,835  valueQuantity
   77.7%   514,835  valueQuantity.system
   77.7%   514,835  valueQuantity.value


In [4]:
print_inventory("Condition")

Condition  (44,368 resources)
   pct     count  path
----------------------------------------------------------------------
  100.0%    44,368  category
  100.0%    44,368  category[].coding
  100.0%    44,368  category[].coding[].code
  100.0%    44,368  category[].coding[].display
  100.0%    44,368  category[].coding[].system
  100.0%    44,368  clinicalStatus
  100.0%    44,368  clinicalStatus.coding
  100.0%    44,368  clinicalStatus.coding[].code
  100.0%    44,368  clinicalStatus.coding[].system
  100.0%    44,368  code
  100.0%    44,368  code.coding
  100.0%    44,368  code.coding[].code
  100.0%    44,368  code.coding[].display
  100.0%    44,368  code.coding[].system
  100.0%    44,368  code.text
  100.0%    44,368  encounter
  100.0%    44,368  encounter.reference
  100.0%    44,368  id
  100.0%    44,368  meta
  100.0%    44,368  meta.profile
  100.0%    44,368  onsetDateTime
  100.0%    44,368  recordedDate
  100.0%    44,368  resourceType
  100.0%    44,368  subject
  10

In [5]:
print_inventory("MedicationRequest", min_pct=5.0)

MedicationRequest  (63,140 resources)
   pct     count  path
----------------------------------------------------------------------
  100.0%    63,140  authoredOn
  100.0%    63,140  category
  100.0%    63,140  category[].coding
  100.0%    63,140  category[].coding[].code
  100.0%    63,140  category[].coding[].display
  100.0%    63,140  category[].coding[].system
  100.0%    63,140  category[].text
  100.0%    63,140  encounter
  100.0%    63,140  encounter.reference
  100.0%    63,140  id
  100.0%    63,140  intent
  100.0%    63,140  meta
  100.0%    63,140  meta.profile
  100.0%    63,140  requester
  100.0%    63,140  requester.display
  100.0%    63,140  requester.reference
  100.0%    63,140  resourceType
  100.0%    63,140  status
  100.0%    63,140  subject
  100.0%    63,140  subject.reference
   73.4%    46,365  reasonReference
   73.4%    46,365  reasonReference[].display
   73.4%    46,365  reasonReference[].reference
   65.6%    41,397  medicationCodeableConcept
   65.

## 2. Generate the MIMIC-IV mapping draft -- WORK IN PROGRESS!!

Writes `references/synthea_to_mimiciv.md` — a structural draft pairing every Synthea field with a proposed MIMIC-IV target.

The cell pre-fills high-confidence rows (resource→table for all 20 types; field-level for `Patient`) and leaves the rest blank to be filled in.
Re-run after editing the maps to regenerate.

**Do not re-run once we begin to manually edit the md file.**

In [6]:
OUTPUT_PATH = Path("../references/synthea_to_mimiciv.md")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# DRAFT — high-confidence resource→table proposals only. Verify with team.
RESOURCE_TABLE_MAP = {
    "Patient": "`mimiciv_hosp.patients`",
    "Encounter": "`mimiciv_hosp.admissions` (+ `mimiciv_icu.icustays` for ICU)",
    "Condition": "`mimiciv_hosp.diagnoses_icd` (SNOMED-CT → ICD-10 mapping required)",
    "Observation": "`mimiciv_hosp.labevents` / `mimiciv_icu.chartevents` (split by category; LOINC → itemid)",
    "MedicationRequest": "`mimiciv_hosp.prescriptions` (RxNorm → GSN/NDC)",
    "MedicationAdministration": "`mimiciv_icu.inputevents` (RxNorm → GSN/NDC)",
    "Medication": "`mimiciv_hosp.prescriptions` (referenced by MedicationRequest; flatten on ingest)",
    "Procedure": "`mimiciv_hosp.procedures_icd` / `mimiciv_icu.procedureevents` (SNOMED-CT → ICD-10-PCS)",
    "DiagnosticReport": "`mimiciv_hosp.labevents` (aggregated lab panels)",
    "Immunization": "*(no direct MIMIC-IV table)*",
    "AllergyIntolerance": "*(no direct MIMIC-IV table)*",
    "CarePlan": "*(no direct MIMIC-IV table)*",
    "CareTeam": "*(no direct MIMIC-IV table — providers folded into `caregiver`)*",
    "Device": "`mimiciv_icu.inputevents` (device-related)",
    "DocumentReference": "MIMIC-IV-Note (`discharge`, `radiology`) — separate dataset",
    "ImagingStudy": "*(out of scope — see MIMIC-CXR)*",
    "Claim": "*(out of scope — billing data not in MIMIC core)*",
    "ExplanationOfBenefit": "*(out of scope — billing data not in MIMIC core)*",
    "Provenance": "*(FHIR metadata — not mapped)*",
    "SupplyDelivery": "*(no direct MIMIC-IV table)*",
}

# DRAFT — Patient is the most universally knowable resource. Verify before relying on this.
PATIENT_FIELD_MAP = {
    "id": ("`patients.subject_id`", "Synthea UUIDs vs. MIMIC int — surrogate-key on load."),
    "gender": ("`patients.gender`", "Both use 'M' / 'F'."),
    "birthDate": ("derived → `patients.anchor_age`, `patients.anchor_year`", "MIMIC-IV anonymizes dates; no raw birthDate is stored."),
    "deceasedDateTime": ("`patients.dod`", "Date-shift applies in real MIMIC; for synthetic cohort, pass through."),
    "address[].city": ("*(none)*", "MIMIC-IV strips geographic identifiers (HIPAA Safe Harbor)."),
    "address[].state": ("*(none)*", "Same — geo PHI removed."),
    "address[].postalCode": ("*(none)*", "Same — geo PHI removed."),
    "name[].family": ("*(none)*", "Names not in MIMIC-IV."),
    "name[].given": ("*(none)*", "Names not in MIMIC-IV."),
    "name[].prefix": ("*(none)*", "Names not in MIMIC-IV."),
    "telecom[].value": ("*(none)*", "Contact info not in MIMIC-IV."),
    "communication[].language.coding[].code": ("`admissions.language`", "Stored at admission level in MIMIC, not patient level."),
    "maritalStatus.coding[].code": ("`admissions.marital_status`", "Admission-level in MIMIC."),
    "extension[].url": ("→ `admissions.race`, `admissions.insurance`, etc.", "Synthea uses us-core-race / us-core-ethnicity extensions on Patient."),
    "multipleBirthBoolean": ("*(none)*", "Not tracked in MIMIC-IV."),
}

FIELD_MAPS = {"Patient": PATIENT_FIELD_MAP}


def render_table(rows, headers):
    out = ["| " + " | ".join(headers) + " |",
           "|" + "|".join("---" for _ in headers) + "|"]
    for row in rows:
        out.append("| " + " | ".join(c if c else "" for c in row) + " |")
    return "\n".join(out)


lines = []
lines.append("# Synthea → MIMIC-IV schema mapping (draft)")
lines.append("")
lines.append("Generated from the cohort in `data/raw/synthea/fhir/` by `notebooks/P1_Wk1_schema_mapping.ipynb` § 2.")
lines.append("")
lines.append("**Status:** DRAFT — review pre-filled rows; fill blank `proposed MIMIC-IV target` cells; flag fields that should be dropped.")
lines.append("")
lines.append("Two distinct mapping problems to solve:")
lines.append("")
lines.append("1. **Structural** — FHIR resource/field → MIMIC-IV table/column (this document).")
lines.append("2. **Vocabulary** — SNOMED-CT / LOINC / RxNorm → ICD-9/10 / itemid / GSN/NDC. Out of scope for this draft; needs separate concept-normalization work.")
lines.append("")
lines.append(f"Cohort scanned: {sum(counts_per_type.values()):,} resources across {len(counts_per_type)} resource types.")
lines.append("")
lines.append("## 1. Resource type → MIMIC-IV table")
lines.append("")
rows = [[rtype, f"{n:,}", RESOURCE_TABLE_MAP.get(rtype, "*(not yet mapped)*")]
        for rtype, n in counts_per_type.most_common()]
lines.append(render_table(rows, ["Resource type", "N", "Proposed MIMIC-IV table"]))
lines.append("")
lines.append("## 2. Field-level mappings")
lines.append("")
lines.append("Each section below lists every JSON path that appears in at least one resource of that type, sorted by frequency. `pct` is the share of resources of that type containing the path.")
lines.append("")

for rtype, _ in counts_per_type.most_common():
    n = counts_per_type[rtype]
    field_map = FIELD_MAPS.get(rtype, {})
    lines.append(f"### {rtype} ({n:,} resources)")
    lines.append("")
    rows = []
    for path, count in sorted(paths_per_type[rtype].items(), key=lambda kv: (-kv[1], kv[0])):
        pct = 100 * count / n
        proposed, notes = field_map.get(path, ("", ""))
        rows.append([f"{pct:.1f}%", f"`{path}`", proposed, notes])
    lines.append(render_table(rows, ["pct", "FHIR path", "proposed MIMIC-IV target", "notes"]))
    lines.append("")

OUTPUT_PATH.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote draft to {OUTPUT_PATH.resolve()}")
print(f"Size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB, {len(lines):,} lines")

Wrote draft to W:\dev\EXAI-KG\references\synthea_to_mimiciv.md
Size: 40.1 KB, 101 lines


## 3. Vocabulary mapping  *(placeholder)*

Concept-normalization work — separate from the structural mapping above. Tasks that potentially live here:

- **SNOMED-CT → ICD-10** for `Condition.code` (44k rows). Likely via UMLS or the SNOMED→ICD-10-CM map distributed by NLM.
- **LOINC → MIMIC `itemid`** for `Observation.code` (663k rows). MIMIC's `d_labitems` / `d_items` carry LOINC where known; gaps will need manual review.
- **RxNorm → GSN/NDC** for `MedicationRequest.medicationCodeableConcept` (41k rows). RxNorm provides RXCUI→NDC crosswalks.
- **SNOMED-CT → ICD-10-PCS** for `Procedure.code` (200k rows). Sparser crosswalk; may require partial coverage + manual review.

Empty for now - TBD.